# nb04 — Two-Stage Tuning + First Submission
**UCU AI 2026 — HW2 — Multiple Object Video Detection (VisDrone)**

## Goal
Satisfy the *Two-stage Anchor-based (1.00 pt)* checkpoint on top of the nb03 baseline.

## What the rubric requires
- ✅ Two training-parameter experiments (RPN proposals, RoI sampling)
- ✅ A tuned model compared to the initial baseline
- ✅ mAP / AP50 / AP_small / inference speed on validation
- ✅ Failure-case analysis of the tuned model
- ✅ A valid Kaggle submission (`submission.csv`) to avoid the "beat the baseline or grade canceled" rule

## Fixed protocol (locked from nb02, do NOT change)
- Preprocessing: `letterbox-1280`
- Optimizer: SGD lr=0.005, momentum=0.9
- Scheduler: StepLR(step=3, gamma=0.1)
- Batch size: 2
- Epochs: 1 (T4 time limit — explained in the report)
- Ignore-aware RPN assignment: on

## Experiments
| ID | Config change | Motivation |
|---|---|---|
| A | `rpn_pre_nms_top_n_train: 1000 → 2000` | Give small (31 px) objects a chance to survive RPN filtering |
| B | `box_positive_fraction: 0.25 → 0.50` | Stronger class signal on rare classes (van/truck confusion) |

## Reference
Baseline (nb03): mAP=**0.1373**, AP50=0.2809, AP_small=0.0557, **4.67 FPS**

## 1. Bootstrap
Rebuild the exact nb03 environment: same common.py, same clip-level split, same full-val COCO GT, same model structure + ignore-aware RPN assigner. Anything different here would break the baseline↔tuned comparison.

In [1]:
# =============================================================================
# nb04 bootstrap — identical environment to nb03
# =============================================================================
import sys, json, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

from torchvision.ops import box_iou
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection._utils import Matcher

# ---- Paths -----------------------------------------------------------------
COMP_DIR     = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
DATASET_DIR  = Path("/kaggle/input/datasets/suchok/my-dataset")
OUTPUTS_DIR  = Path("/kaggle/input/datasets/suchok/outputs")           # nb02 artifacts
WEIGHTS_DIR  = Path("/kaggle/input/datasets/suchok/baseline-weights")  # baseline ckpt
WORK_DIR     = Path("/kaggle/working")

assert COMP_DIR.exists(),    f"missing competition: {COMP_DIR}"
assert DATASET_DIR.exists(), f"missing my-dataset: {DATASET_DIR}"
assert OUTPUTS_DIR.exists(), f"missing outputs (nb02): {OUTPUTS_DIR}"
assert WEIGHTS_DIR.exists(), f"missing baseline-weights: {WEIGHTS_DIR}"

WORK_DIR.mkdir(parents=True, exist_ok=True)

# ---- common.py from my-dataset --------------------------------------------
sys.path.insert(0, str(DATASET_DIR))
import common, importlib
importlib.reload(common)
from common import (
    set_seeds, SEED, CLASS_NAMES, NUM_CLASSES, IMAGENET_MEAN, IMAGENET_STD,
    load_splits, index_split, filter_by_clips,
    load_yolo_labels, load_ignore_regions,
    build_fullval_gt, evaluate_full_val, benchmark_speed,
    predict_with_strategy, save_metrics, save_checkpoint,
    detection_collate, VisDroneDetectionDataset,
    predict_test_and_write_submission, read_image_sizes,
    coco_eval_summary,
)

set_seeds(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| torch:", torch.__version__)

# ---- Locked protocol from nb02 --------------------------------------------
chosen = json.load(open(OUTPUTS_DIR / "chosen_preprocessing.json"))
STRAT  = chosen["strategy"]
SIZE   = int(chosen["size"])
print("Locked preprocessing:", STRAT, SIZE)

train_clips, val_clips = load_splits(OUTPUTS_DIR / "splits")
assert set(train_clips).isdisjoint(set(val_clips))
print(f"clips: train={len(train_clips)}  val={len(val_clips)}")

TRAIN_DIR = COMP_DIR / "train"
VAL_DIR   = COMP_DIR / "val"
TEST_CSV  = COMP_DIR / "test_images.csv"

train_index = filter_by_clips(index_split(TRAIN_DIR), train_clips)
val_index   = filter_by_clips(index_split(VAL_DIR),   val_clips)
print(f"frames: train={len(train_index):,}  val={len(val_index):,}")

# ---- Full-val GT (reuse published copy, else rebuild) ---------------------
gt_fullval_ignore   = WORK_DIR / "gt_fullval_ignore.json"
gt_fullval_noignore = WORK_DIR / "gt_fullval_noignore.json"

for src_dir in (WEIGHTS_DIR, OUTPUTS_DIR):
    for name in ("gt_fullval_ignore.json", "gt_fullval_noignore.json"):
        if (src_dir / name).exists() and not (WORK_DIR / name).exists():
            shutil.copy(src_dir / name, WORK_DIR / name)
            print(f"copied {name} from {src_dir}")

if not gt_fullval_ignore.exists():
    print("rebuilding gt_fullval_ignore.json ...")
    build_fullval_gt(val_index, str(gt_fullval_ignore), include_ignore=True)
if not gt_fullval_noignore.exists():
    print("rebuilding gt_fullval_noignore.json ...")
    build_fullval_gt(val_index, str(gt_fullval_noignore), include_ignore=False)

# ---- Model builder (parameterised for tuning) -----------------------------
def build_fasterrcnn(rpn_pre_nms_train=1000, rpn_post_nms_train=1000,
                     rpn_pre_nms_test=1000,  rpn_post_nms_test=300,
                     box_positive_fraction=0.25, box_batch_size=512,
                     score_thresh=0.05, nms_thresh=0.5):
    m = fasterrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None,
        min_size=SIZE, max_size=SIZE,
        rpn_pre_nms_top_n_train=rpn_pre_nms_train,
        rpn_post_nms_top_n_train=rpn_post_nms_train,
        rpn_pre_nms_top_n_test=rpn_pre_nms_test,
        rpn_post_nms_top_n_test=rpn_post_nms_test,
        box_batch_size_per_image=box_batch_size,
        box_positive_fraction=box_positive_fraction,
        box_score_thresh=score_thresh,
        box_nms_thresh=nms_thresh,
    )
    in_feat = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NUM_CLASSES + 1)
    return m

# ---- IgnoreAwareAssigner (identical to nb03 §6) ---------------------------
class IgnoreAwareAssigner(torch.nn.Module):
    def __init__(self, high_thresh=0.5, low_thresh=0.4, ignore_iou_thresh=0.3):
        super().__init__()
        self.matcher = Matcher(high_thresh, low_thresh, allow_low_quality_matches=True)
        self.ignore_iou_thresh = ignore_iou_thresh
        self.current_ignore_boxes = None

    def forward(self, anchors, gt_boxes):
        matched_labels, matched_boxes = self.matcher(anchors, gt_boxes)
        if self.current_ignore_boxes is None or len(self.current_ignore_boxes) == 0:
            return matched_labels, matched_boxes
        if len(gt_boxes) == 0:
            return matched_labels, matched_boxes
        matched_labels = matched_labels.clone()
        iou = box_iou(anchors, self.current_ignore_boxes).max(dim=1).values
        ignored = (iou > self.ignore_iou_thresh) & (matched_labels >= 0)
        matched_labels[ignored] = -1
        return matched_labels, matched_boxes

BASELINE_CKPT = WEIGHTS_DIR / "fasterrcnn_baseline_last.pth"
assert BASELINE_CKPT.exists(), f"missing checkpoint: {BASELINE_CKPT}"

def load_baseline_weights(m, ckpt=BASELINE_CKPT, device=DEVICE):
    """Attach ignore-aware assigner, then load baseline weights with strict=False."""
    m.rpn.box_assigner = IgnoreAwareAssigner().to(device)
    state = torch.load(ckpt, map_location=device)
    missing, unexpected = m.load_state_dict(state, strict=False)
    # Filter expected noise from the assigner state
    missing    = [k for k in missing    if "box_assigner" not in k]
    unexpected = [k for k in unexpected if "box_assigner" not in k]
    if missing:    print("WARN missing keys:",    missing[:5], "...")
    if unexpected: print("WARN unexpected keys:", unexpected[:5], "...")
    m.to(device)
    print(f"loaded baseline checkpoint: {ckpt.name}")
    return m

# ---- W&B ----------------------------------------------------------------
import wandb
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("W&B login OK")
except Exception as e:
    print(f"WARN W&B secret unavailable: {e}")

print("=" * 60)
print("bootstrap OK — run cells 3+")
print("=" * 60)

DEVICE: cuda | torch: 2.11.0+cu130
Locked preprocessing: letterbox 1280
clips: train=56  val=7
frames: train=24,201  val=2,846
copied gt_fullval_ignore.json from /kaggle/input/datasets/suchok/baseline-weights
copied gt_fullval_noignore.json from /kaggle/input/datasets/suchok/baseline-weights


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: suchok-pn (suchok-pn-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login OK
bootstrap OK — run cells 3+


## 2. Baseline sanity check
Reload the nb03 weights into a freshly built baseline-config model and re-evaluate on full val. The numbers must match nb03 (mAP ≈ 0.1373). If they don't, the bootstrap is not equivalent and tuning results are meaningless.

In [2]:
set_seeds(SEED)
model_base = load_baseline_weights(build_fasterrcnn()).eval()

metrics_base, pred_base_json = evaluate_full_val(
    model_base, val_index, STRAT, SIZE, str(gt_fullval_ignore),
    score_thr=0.001, device=DEVICE,
    out_json=str(WORK_DIR / "val_preds_baseline_recheck.json"))
print("baseline recheck:", json.dumps(metrics_base, indent=2))

NB03_MAP = 0.1373
delta = abs(metrics_base["mAP"] - NB03_MAP)
print(f"|ΔmAP| vs nb03 = {delta:.4f}  (expected < 0.002)")

# Free VRAM before training
del model_base
torch.cuda.empty_cache()

loaded baseline checkpoint: fasterrcnn_baseline_last.pth


pred letterbox-1280:   0%|          | 0/2846 [00:00<?, ?it/s]

baseline recheck: {
  "mAP": 0.14749652689171325,
  "AP50": 0.30937599997657345,
  "AP75": 0.12673569937517382,
  "AP_small": 0.08031522658266038,
  "AP_medium": 0.1570005813142584,
  "AP_large": 0.2666030957485943
}
|ΔmAP| vs nb03 = 0.0102  (expected < 0.002)


## 3. Training loop
Identical to nb03: SGD + StepLR, batch=2, 1 epoch. Logs every 100 steps to W&B.

In [3]:
from torch.utils.data import DataLoader
from torch.optim import SGD
from torch.optim.lr_scheduler import StepLR

def train_one_epoch(model, loader, optimizer, scheduler, device,
                    wandb_run=None, log_every=100):
    model.train()
    losses_log = {"loss_classifier": [], "loss_box_reg": [],
                  "loss_objectness": [], "loss_rpn_box_reg": []}
    t0 = time.time()
    for step, (images, targets) in enumerate(loader):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        for k in losses_log:
            losses_log[k].append(float(loss_dict[k]))

        if step % log_every == 0:
            total = sum(v[-1] for v in losses_log.values())
            print(f"step {step:4d} | total {total:.4f} | "
                  + " ".join(f"{k}={v[-1]:.3f}" for k, v in losses_log.items())
                  + f" | {time.time()-t0:.0f}s")
            if wandb_run:
                wandb_run.log({"train/total": total,
                               **{f"train/{k}": v[-1] for k, v in losses_log.items()},
                               "train/step": step})
    return {k: float(np.mean(v)) for k, v in losses_log.items()}

## 4. Experiment A — RPN pre-NMS proposals 1000 → 2000
**Motivation:** RPN retains only the top-1000 proposals per image during training. A 31 px pedestrian competes poorly against large road/vegetation anchors. Doubling the proposal budget should raise RPN recall on small objects, directly targeting `AP_small=0.0557`.
**Expected:** AP_small ↑, mAP ↑ modestly, FPS unchanged (test-time top-N is separate).

In [4]:
set_seeds(SEED)
run_a = wandb.init(project="ucu-hw2-detection",
                   name="frnn_tuneA_rpn2000",
                   config={"rpn_pre_nms_top_n_train": 2000,
                           "rpn_post_nms_top_n_train": 1000,
                           "box_positive_fraction": 0.25,
                           "epochs": 1, "lr": 0.005, "batch": 2,
                           "preproc": f"{STRAT}-{SIZE}"},
                   reinit=True)
print("W&B A:", run_a.url)

model_a = load_baseline_weights(build_fasterrcnn(rpn_pre_nms_train=2000))

ds_train = VisDroneDetectionDataset(train_index, STRAT, SIZE, is_train=False)
loader_a = DataLoader(ds_train, batch_size=2, shuffle=True,
                      num_workers=2, collate_fn=detection_collate)

opt_a = SGD([p for p in model_a.parameters() if p.requires_grad],
            lr=0.005, momentum=0.9)
sch_a = StepLR(opt_a, step_size=3, gamma=0.1)

losses_a = train_one_epoch(model_a, loader_a, opt_a, sch_a, DEVICE, run_a)
print("mean train losses A:", losses_a)

save_checkpoint(model_a, WORK_DIR / "fasterrcnn_tuneA_rpn2000.pth")
run_a.log({"train/mean_losses": losses_a})

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run mq3z3goe
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261105_174108-mq3z3goe
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run frnn_tuneA_rpn2000
wandb: ⭐️ View project at https://wandb.ai/suchok-pn-ukrainian-catholic-university/ucu-hw2-detection
wandb: 🚀 View run at https://wandb.ai/suchok-pn-ukrainian-catholic-university/ucu-hw2-detection/runs/mq3z3goe


W&B A: https://wandb.ai/suchok-pn-ukrainian-catholic-university/ucu-hw2-detection/runs/mq3z3goe
loaded baseline checkpoint: fasterrcnn_baseline_last.pth


/tmp/ipykernel_23/1396117594.py:24: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses_log[k].append(float(loss_dict[k]))


step    0 | total 0.9179 | loss_classifier=0.375 loss_box_reg=0.359 loss_objectness=0.048 loss_rpn_box_reg=0.135 | 3s
step  100 | total 0.8019 | loss_classifier=0.285 loss_box_reg=0.317 loss_objectness=0.072 loss_rpn_box_reg=0.129 | 179s
step  200 | total 1.0436 | loss_classifier=0.454 loss_box_reg=0.462 loss_objectness=0.024 loss_rpn_box_reg=0.103 | 356s
step  300 | total 0.9752 | loss_classifier=0.444 loss_box_reg=0.415 loss_objectness=0.032 loss_rpn_box_reg=0.083 | 533s
step  400 | total 0.8721 | loss_classifier=0.279 loss_box_reg=0.420 loss_objectness=0.053 loss_rpn_box_reg=0.119 | 709s
step  500 | total 0.8606 | loss_classifier=0.304 loss_box_reg=0.345 loss_objectness=0.071 loss_rpn_box_reg=0.141 | 886s
step  600 | total 0.7675 | loss_classifier=0.289 loss_box_reg=0.358 loss_objectness=0.054 loss_rpn_box_reg=0.067 | 1062s
step  700 | total 0.6305 | loss_classifier=0.238 loss_box_reg=0.318 loss_objectness=0.051 loss_rpn_box_reg=0.023 | 1238s
step  800 | total 0.9382 | loss_classifi

### 4.1 Evaluate A on full val
Standard COCO metrics + speed. This is the number that goes into the comparison table.

In [5]:
model_a.eval()
metrics_a, pred_a_json = evaluate_full_val(
    model_a, val_index, STRAT, SIZE, str(gt_fullval_ignore),
    score_thr=0.001, device=DEVICE,
    out_json=str(WORK_DIR / "val_preds_A.json"))
print("Variant A metrics:", json.dumps(metrics_a, indent=2))

speed_a = benchmark_speed(model_a, val_index, STRAT, SIZE, n=50, device=DEVICE)
print("Variant A speed:", speed_a)

run_a.log({"val/mAP": metrics_a["mAP"], "val/AP50": metrics_a["AP50"],
           "val/AP75": metrics_a["AP75"],
           "val/AP_small": metrics_a["AP_small"],
           "val/AP_medium": metrics_a["AP_medium"],
           "val/AP_large": metrics_a["AP_large"],
           "speed/gpu_fps": speed_a["gpu_fps"] or 0})
run_a.finish()

del model_a
torch.cuda.empty_cache()

pred letterbox-1280:   0%|          | 0/2846 [00:00<?, ?it/s]

Variant A metrics: {
  "mAP": 0.15612895963474377,
  "AP50": 0.3233825797467062,
  "AP75": 0.13351126774659625,
  "AP_small": 0.09629251255084864,
  "AP_medium": 0.16556544121326644,
  "AP_large": 0.27058399571028974
}


RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor